In [42]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn import tree 
from sklearn import ensemble 
from sklearn import metrics 
import optuna
from catboost import CatBoostClassifier

In [43]:
#Прочитаем обработанные данные
X_train = pd.read_csv('../data/X_train_scaled.csv')
X_test = pd.read_csv('../data/X_test_scaled.csv')

y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [44]:
#Инициализируем функцию для подсчета метрик моделей
results = []

def evaluate_model(name, model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    results.append({
        'model': name,
        'accuracy': metrics.accuracy_score(y_test, y_pred),
        'precision': metrics.precision_score(y_test, y_pred),
        'f1': metrics.f1_score(y_test, y_pred),
        'roc_auc': metrics.roc_auc_score(y_test, y_proba)
    })

In [45]:
#Попробуем построить рандомный лес
#Инициализируем лес
rf = ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    min_samples_leaf=5,
    max_depth=10,
    random_state=42
)
#Обучаем лес
rf.fit(X_train, y_train)
evaluate_model('RandomForest', rf, X_test, y_test)
#Делаем предсказание
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.692
Precision: 0.712
F1: 0.63
ROC-AUC: 0.746


Случайный лес справляется с задачей лучше, чем модели в бейзлайне, попробуем градиентный бустинг

In [46]:
#Инициализируем градиентный бустинг
gb = ensemble.GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=300,
    min_samples_leaf=5,
    max_depth=5,
    random_state=42
)
#Обучаем бустинг
gb.fit(X_train, y_train)
evaluate_model('Boosting', gb, X_test, y_test)
#Делаем предсказание
y_pred = gb.predict(X_test)
y_proba = gb.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.689
Precision: 0.704
F1: 0.627
ROC-AUC: 0.742


Совсем немного, но хуже, попробуем стекинг с логрегрессией, деревом и бустингом

In [47]:
#Инициализируем регрессию
lr = LogisticRegression(
    solver='sag',
    random_state=42,
    max_iter=1000
)
#Инициализируем дерево
tree_model = tree.DecisionTreeClassifier(
    random_state=42,
    criterion='entropy'
)

In [48]:
#Делаем стэккинг с мета-моделью логистической регрессией
stack = ensemble.StackingClassifier(
    estimators=[
        ('lr', lr),
        ('tree', tree_model),
        ('gb', gb)
    ],
    final_estimator=LogisticRegression()
)
#Обучаем стеккинг
stack.fit(X_train, y_train)
evaluate_model('Stacking', stack, X_test, y_test)
#делаем предсказание
y_pred = stack.predict(X_test)
y_proba = stack.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.688
Precision: 0.694
F1: 0.633
ROC-AUC: 0.746


Практически без изменений

In [49]:
importances = gb.feature_importances_
feature_importance = pd.Series(
    gb.feature_importances_,
    index=X_train.columns
)

feature_importance.sort_values(ascending=False).head(3)

13    0.249454
0     0.230470
8     0.096655
dtype: float64

Попробуем найти гиперпараметры с помощью ***optuna***

In [50]:
def optuna_rf(trial):
    #задаем пространства поиска гиперпараметров
    n_estimators = trial.suggest_int('n_estimators', 100, 200, 1)
    max_depth = trial.suggest_int('max_depth', 10, 30, 1)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10, 1)

    #создаем модель
    model = ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    #обучаем модель
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = metrics.f1_score(y_test, y_pred)

    return f1

In [51]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_rf, n_trials=50)

[I 2026-03-07 16:40:52,750] A new study created in memory with name: no-name-34f9a871-abd1-43e2-b104-fe8a457f265b
[I 2026-03-07 16:40:52,940] Trial 0 finished with value: 0.6294267981014969 and parameters: {'n_estimators': 111, 'max_depth': 10, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.6294267981014969.
[I 2026-03-07 16:40:53,306] Trial 1 finished with value: 0.612987012987013 and parameters: {'n_estimators': 194, 'max_depth': 29, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.6294267981014969.
[I 2026-03-07 16:40:53,770] Trial 2 finished with value: 0.6147747747747748 and parameters: {'n_estimators': 183, 'max_depth': 23, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.6294267981014969.
[I 2026-03-07 16:40:54,185] Trial 3 finished with value: 0.6162665711214619 and parameters: {'n_estimators': 163, 'max_depth': 24, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.6294267981014969.
[I 2026-03-07 16:40:54,476] Trial 4 finished with value: 0.612426035502958

In [52]:
print(study.best_params)

{'n_estimators': 144, 'max_depth': 11, 'min_samples_leaf': 5}


In [53]:
meta_model = ensemble.RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

meta_model.fit(X_train, y_train)
evaluate_model('Meta_model', meta_model, X_test, y_test)
y_pred = meta_model.predict(X_test)
y_proba = meta_model.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.694
Precision: 0.712
F1: 0.634
ROC-AUC: 0.745


In [54]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    depth=6,
    learning_rate=0.05,
    iterations=500,
    random_state=42,
    verbose=False
)

cat.fit(X_train, y_train)
evaluate_model('CatBoost', cat, X_test, y_test)
pred = cat.predict(X_test)
y_proba = cat.predict_proba(X_test)[:, 1]

print('Accuracy:', round(metrics.accuracy_score(y_test, y_pred), 3))
print('Precision:', round(metrics.precision_score(y_test, y_pred), 3))
print('F1:', round(metrics.f1_score(y_test, y_pred), 3))
print('ROC-AUC:', round(metrics.roc_auc_score(y_test, y_proba), 3))

Accuracy: 0.694
Precision: 0.712
F1: 0.634
ROC-AUC: 0.744


In [55]:
results_df = pd.DataFrame(results)

results_df.round(3).sort_values(by='f1', ascending=False)

,model,accuracy,precision,f1,roc_auc
3,Meta_model,0.694,0.712,0.634,0.745
2,Stacking,0.688,0.694,0.633,0.746
0,RandomForest,0.692,0.712,0.630,0.746
1,Boosting,0.689,0.704,0.627,0.742
4,CatBoost,0.688,0.703,0.627,0.744
